# SMS Spam — Text Classification Benchmark (Dataset #4)

**Domain:** Text | **Task:** Classify SMS as spam or ham

Workflow: **baseline → primary → one advanced → single comparison table → export → resume bullet.**

- **Baseline:** most-frequent predictor (DummyClassifier)
- **Primary:** Multinomial Naive Bayes + TF-IDF
- **Advanced:** Linear SVM + TF-IDF, Logistic Regression + TF-IDF
- **Metrics:** Accuracy, F1, ROC-AUC (5-fold CV on train, stratified hold-out test)

> Data: SMS Spam Collection (5574 rows). Loaded from a public TSV mirror identical to the
> [Kaggle `uciml/sms-spam-collection-dataset`](https://www.kaggle.com/datasets/uciml/sms-spam-collection-dataset).

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.dummy import DummyClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [2]:
# Load SMS Spam (public TSV mirror == Kaggle dataset)
URL = "https://raw.githubusercontent.com/justmarkham/DAT8/master/data/sms.tsv"
df = pd.read_csv(URL, sep="\t", header=None, names=["label", "message"])
df["label_num"] = (df["label"] == "spam").astype(int)

import os
os.makedirs("data", exist_ok=True)
df.to_csv("data/sms-spam.csv", index=False)  # save raw copy for the Streamlit app

print(f"Rows: {len(df)}, Cols: {df.shape[1]}")
df.head()

Rows: 5572, Cols: 3


,label,message,label_num
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


## 1. EDA & Preprocessing

In [3]:
# Class imbalance + missing + message length
print("Class balance:")
print(df["label"].value_counts(normalize=True).round(3))
print()
print("Missing:", df.isna().sum().sum())
df["len"] = df["message"].str.len()
df.groupby("label")["len"].describe().round(1)

Class balance:
label
ham     0.866
spam    0.134
Name: proportion, dtype: float64

Missing: 0


,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
ham,4825.0,71.5,58.4,2.0,33.0,52.0,93.0,910.0
spam,747.0,138.7,28.9,13.0,133.0,149.0,157.0,223.0


In [4]:
# Split text/label
X = df["message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 4457  |  Test: 1115


In [5]:
# Baseline: always predict the majority class (ham)
baseline = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
y_base = baseline.predict(X_test)
print(f"Baseline  Acc={accuracy_score(y_test, y_base):.3f}  "
      f"F1={f1_score(y_test, y_base):.3f}")

Baseline  Acc=0.866  F1=0.000


## 2. Primary — Multinomial Naive Bayes + TF-IDF

In [6]:
def make_text_model(clf):
    return Pipeline([
        ("tfidf", TfidfVectorizer(stop_words="english", ngram_range=(1, 2),
                                  min_df=2, max_df=0.9)),
        ("clf", clf),
    ])

def evaluate(name, pipe):
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    if hasattr(pipe, "predict_proba"):
        y_score = pipe.predict_proba(X_test)[:, 1]
    else:  # LinearSVC has no predict_proba
        y_score = pipe.decision_function(X_test)
    cv_acc = cross_val_score(pipe, X_train, y_train, cv=5, scoring="accuracy").mean()
    row = {
        "model": name,
        "cv_accuracy": cv_acc,
        "test_accuracy": accuracy_score(y_test, y_pred),
        "test_f1": f1_score(y_test, y_pred),
        "test_auc": roc_auc_score(y_test, y_score),
    }
    cv_rows.append(row)
    print(f"{name:24s} CV-Acc={cv_acc:.3f}  Acc={row['test_accuracy']:.3f}  "
          f"F1={row['test_f1']:.3f}  AUC={row['test_auc']:.3f}")

cv_rows = []
nb = make_text_model(MultinomialNB())
evaluate("Naive Bayes + TF-IDF", nb)

Naive Bayes + TF-IDF     CV-Acc=0.968  Acc=0.970  F1=0.875  AUC=0.988


## 3. Advanced — Linear SVM, Logistic Regression

In [7]:
svm = make_text_model(LinearSVC(max_iter=5000, random_state=RANDOM_STATE))
evaluate("Linear SVM + TF-IDF", svm)

logreg = make_text_model(LogisticRegression(max_iter=2000))
evaluate("LogReg + TF-IDF", logreg)

Linear SVM + TF-IDF      CV-Acc=0.984  Acc=0.980  F1=0.921  AUC=0.989


LogReg + TF-IDF          CV-Acc=0.961  Acc=0.970  F1=0.875  AUC=0.986


## 4. Comparison Table (all methods)

In [8]:
results = pd.DataFrame(cv_rows).sort_values(
    ["test_accuracy", "test_f1", "cv_accuracy"], ascending=False).reset_index(drop=True)
results.insert(0, "rank", results.index + 1)
results.round(3)

,rank,model,cv_accuracy,test_accuracy,test_f1,test_auc
0,1,Linear SVM + TF-IDF,0.984,0.98,0.921,0.989
1,2,Naive Bayes + TF-IDF,0.968,0.97,0.875,0.988
2,3,LogReg + TF-IDF,0.961,0.97,0.875,0.986


In [9]:
best = results.iloc[0]
print("BEST MODEL:", best["model"])
print(f"  CV-Acc={best['cv_accuracy']:.3f} | Acc={best['test_accuracy']:.3f} | "
      f"F1={best['test_f1']:.3f} | AUC={best['test_auc']:.3f}")

BEST MODEL: Linear SVM + TF-IDF
  CV-Acc=0.984 | Acc=0.980 | F1=0.921 | AUC=0.989


## 5. Export Best Model (for Streamlit deployment)

In [10]:
import joblib, json

# Refit the winning pipeline on ALL data for deployment (tfidf is part of the pipeline)
fitted_lookup = {"Naive Bayes + TF-IDF": nb, "Linear SVM + TF-IDF": svm, "LogReg + TF-IDF": logreg}
best_obj = fitted_lookup[best["model"]]
best_obj.fit(X, y)  # refit on full dataset (metrics above stay hold-out based)

os.makedirs("models", exist_ok=True)
joblib.dump(best_obj, "models/best_model.joblib")
results.to_csv("models/comparison.csv", index=False)

meta = {
    "dataset": "SMS Spam",
    "target": "label (spam/ham)",
    "best_model": best["model"],
    "has_proba": bool(hasattr(best_obj, "predict_proba")),
    "metrics": {
        "cv_accuracy": round(float(best["cv_accuracy"]), 3),
        "test_accuracy": round(float(best["test_accuracy"]), 3),
        "test_f1": round(float(best["test_f1"]), 3),
        "test_auc": round(float(best["test_auc"]), 3),
    },
    "n_samples": int(len(df)),
    "input": "raw SMS text",
    "loader": "joblib.load('models/best_model.joblib')",
}
with open("models/model_metadata.json", "w") as f:
    json.dump(meta, f, indent=2)

print("Saved models/best_model.joblib + models/comparison.csv + models/model_metadata.json")
print(json.dumps(meta, indent=2))

Saved models/best_model.joblib + models/comparison.csv + models/model_metadata.json
{
  "dataset": "SMS Spam",
  "target": "label (spam/ham)",
  "best_model": "Linear SVM + TF-IDF",
  "has_proba": false,
  "metrics": {
    "cv_accuracy": 0.984,
    "test_accuracy": 0.98,
    "test_f1": 0.921,
    "test_auc": 0.989
  },
  "n_samples": 5572,
  "input": "raw SMS text",
  "loader": "joblib.load('models/best_model.joblib')"
}


## 6. Resume Bullet

In [11]:
bullet = (
    f"Benchmarked {len(results)} text-classification methods on SMS Spam ({len(df)} messages, "
    f"TF-IDF word 1-2 grams); achieved {best['model']} at test accuracy "
    f"{best['test_accuracy']:.3f} (F1 {best['test_f1']:.3f}, AUC {best['test_auc']:.3f}) "
    f"+{best['test_accuracy'] - 0.866:.3f} accuracy vs baseline."
)
print("Resume bullet (SMS Spam):")
print("\u2022 " + bullet)

Resume bullet (SMS Spam):
• Benchmarked 3 text-classification methods on SMS Spam (5572 messages, TF-IDF word 1-2 grams); achieved Linear SVM + TF-IDF at test accuracy 0.980 (F1 0.921, AUC 0.989) +0.114 accuracy vs baseline.
